# Adia Lab Causal Discovery — 1st Place Solution

Faithful reimplementation of [thetourney.github.io/adia-report](https://thetourney.github.io/adia-report/).

**Key difference from a naive approach**: the 3rd channel is a **multivariate**
kernel regression coefficient — predicting variable j from *all* other variables
simultaneously (not just pairwise), using a Gaussian kernel on full-row distance.
This captures conditional dependencies critical for causal discovery.

### Setup

In [1]:
# %pip install pytorch_lightning crunch-cli --upgrade --quiet
# Uncomment the line below for CrunchDAO notebook setup:
!crunch --env staging setup-notebook causality-discovery 0LieCWz8NrSdOTsZFGE39SaV --no-data

environment: forcing staging urls, ignoring $API_BASE_URL and $WEB_BASE_URL
crunch-cli, version 11.0.0



---
Your token seems to have expired or is invalid.

Please follow this link to copy and paste your new setup command:
https://hub.crunchdao.io/competitions/causality-discovery/submit

If you think that is an error, please contact an administrator.


In [2]:
%env API_BASE_URL=https://api.hub.crunchdao.io/
%env WEB_BASE_URL=https://hub.crunchdao.io/

import crunch.store
crunch.store.load_from_env()

env: API_BASE_URL=https://api.hub.crunchdao.io/
env: WEB_BASE_URL=https://hub.crunchdao.io/


### Imports

In [3]:
import typing
import os
from tqdm.auto import tqdm

import pandas as pd
import numpy as np

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

import pytorch_lightning as pl

import networkx as nx
from sklearn.model_selection import train_test_split

### Configuration

In [4]:
# @crunch/keep:on
N_OBS = 1000
N_KERNEL = 150
N_CLASSES = 8
N_EDGE_TYPES = 7
D_MODEL = 64
CLASS_NAMES = [
    "Confounder", "Collider", "Mediator",
    "Cause of X", "Cause of Y",
    "Consequence of X", "Consequence of Y",
    "Independent",
]

### Graph Utilities

In [5]:
# @crunch/keep:on

def graph_nodes_representation(graph, nodelist):
    adjacency_matrix = nx.adjacency_matrix(graph, nodelist=nodelist).todense()
    return tuple(adjacency_matrix.flatten())

def create_graph_label():
    graph_label = {
        nx.DiGraph([("X", "Y"), ("v", "X"), ("v", "Y")]): "Confounder",
        nx.DiGraph([("X", "Y"), ("X", "v"), ("Y", "v")]): "Collider",
        nx.DiGraph([("X", "Y"), ("X", "v"), ("v", "Y")]): "Mediator",
        nx.DiGraph([("X", "Y"), ("v", "X")]): "Cause of X",
        nx.DiGraph([("X", "Y"), ("v", "Y")]): "Cause of Y",
        nx.DiGraph([("X", "Y"), ("X", "v")]): "Consequence of X",
        nx.DiGraph([("X", "Y"), ("Y", "v")]): "Consequence of Y",
        nx.DiGraph({"X": ["Y"], "v": []}): "Independent",
    }
    nodelist = ["v", "X", "Y"]
    adjacency_label = {
        graph_nodes_representation(graph, nodelist): label
        for graph, label in graph_label.items()
    }
    return graph_label, adjacency_label

_GRAPH_LABEL, _ADJACENCY_LABEL = None, None

def get_adjacency_label():
    global _GRAPH_LABEL, _ADJACENCY_LABEL
    if _ADJACENCY_LABEL is None:
        _GRAPH_LABEL, _ADJACENCY_LABEL = create_graph_label()
    return _ADJACENCY_LABEL

def get_labels(adjacency_matrix, adjacency_label):
    result = {}
    for variable in adjacency_matrix.columns.drop(["X", "Y"]):
        submatrix = adjacency_matrix.loc[
            [variable, "X", "Y"], [variable, "X", "Y"]
        ]
        key = tuple(submatrix.values.flatten())
        result[variable] = adjacency_label[key]
    return result

def transform_proba_to_DAG(nodes, pred):
    G = nx.DiGraph()
    G.add_nodes_from(nodes)
    G.add_edge("X", "Y")
    x_index, y_index = np.unravel_index(
        np.argsort(pred.ravel())[::-1], pred.shape
    )
    for i, j in zip(x_index, y_index):
        n1, n2 = nodes[i], nodes[j]
        if i == j:
            continue
        if {n1, n2} == {"X", "Y"}:
            continue
        if pred[i, j] > 0.5:
            G.add_edge(n1, n2)
            if not nx.is_directed_acyclic_graph(G):
                G.remove_edge(n1, n2)
    return nx.to_numpy_array(G)

### Data Preprocessing — Multivariate Kernel Regression

The report specifies: for edge u→v (variable k → variable j), channel 3 holds
the coefficient $c_{i, k \to j}$ from a **multivariate** local linear regression
predicting variable $j$ from **all** other variables, with Gaussian weights on
the full-row distance.

This is **not** a pairwise regression — it conditions on all variables simultaneously.

In [6]:
def _edge_type(u_name, v_name):
    """Edge type encoding (7 types) as described in the report."""
    uX, uY = u_name == "X", u_name == "Y"
    vX, vY = v_name == "X", v_name == "Y"
    if uX and not vY:  return 0   # u=X, v≠Y
    if uX and vY:      return 1   # u=X, v=Y
    if uY and not vX:  return 2   # u=Y, v≠X
    if uY and vX:      return 3   # u=Y, v=X
    if not uX and not uY and vX: return 4  # u∉{X,Y}, v=X
    if not uX and not uY and vY: return 5  # u∉{X,Y}, v=Y
    return 6                                # none of the above

def compute_multivariate_kernel_coefficients(
    data: np.ndarray,
    n_sub: int = None,
    bandwidth: float = 0.5,
) -> np.ndarray:
    """
    Multivariate kernel regression for ALL target variables at once.

    For each target variable j and each (subsampled) observation i, solve:
        min_c  Σ_l w(x_i, x_l) * (x_{l,j} - c_0 - Σ_{k≠j} c_{k→j} * x_{l,k})²

    where w is a Gaussian kernel on full-row distance.

    Returns:
        coeff_map: dict mapping (k, j) -> np.ndarray of shape (N,)
            coeff_map[(k, j)][i] = c_{i, k→j}
            (interpolated back to all N observations via nearest-neighbor)
    """
    if n_sub is None:
        n_sub = N_KERNEL

    N, p = data.shape
    n_sub = min(n_sub, N)

    # --- Subsample ---
    sub_idx = np.random.choice(N, n_sub, replace=False)
    data_sub = data[sub_idx]  # (n_sub, p)

    # --- Pairwise squared distances in full variable space ---
    diff = data_sub[:, None, :] - data_sub[None, :, :]  # (n_sub, n_sub, p)
    sq_dist = (diff ** 2).sum(axis=-1)                    # (n_sub, n_sub)
    W = np.exp(-sq_dist / (2 * bandwidth ** 2))           # (n_sub, n_sub)

    # --- For each target variable j, solve WLS for all subsampled obs ---
    # Store: for target j, coefficients (n_sub, p) where
    #   col 0 = intercept, cols 1..p-1 = coeffs for other_cols
    all_coeffs = {}  # j -> (n_sub, p) array

    for j in range(p):
        other_cols = [k for k in range(p) if k != j]
        # Design matrix: intercept + other variables
        X_design = np.concatenate(
            [np.ones((n_sub, 1)), data_sub[:, other_cols]], axis=1
        )  # (n_sub, p)  — p = 1 + (p-1)
        y_target = data_sub[:, j]  # (n_sub,)

        # Batch WLS: for each obs i, A_i = X^T diag(W[i]) X, b_i = X^T (W[i]*y)
        # Vectorized using einsum:
        # A_all[i,a,b] = Σ_l W[i,l] * X[l,a] * X[l,b]
        A_all = np.einsum('il,la,lb->iab', W, X_design, X_design)  # (n_sub, p, p)
        # b_all[i,a] = Σ_l W[i,l] * X[l,a] * y[l]
        b_all = np.einsum('il,la,l->ia', W, X_design, y_target)    # (n_sub, p)

        # Regularize and solve
        reg = 1e-6 * np.eye(p)[None, :, :]  # (1, p, p)
        try:
            # c_all = np.linalg.solve(A_all + reg, b_all)  # (n_sub, p)
            c_all = np.linalg.solve(A_all + reg, b_all[:, :, None]).squeeze(-1)
        except np.linalg.LinAlgError:
            c_all = np.zeros((n_sub, p))

        all_coeffs[j] = (c_all, other_cols)

    # --- Interpolate to all N observations via nearest-neighbor ---
    # Find closest subsampled point for each observation
    # Use squared distance in full variable space
    dist_to_sub = np.sum(
        (data[:, None, :] - data_sub[None, :, :]) ** 2, axis=-1
    )  # (N, n_sub)
    nearest = np.argmin(dist_to_sub, axis=1)  # (N,)

    # --- Build coeff_map: (k, j) -> (N,) ---
    coeff_map = {}
    for j in range(p):
        c_all, other_cols = all_coeffs[j]
        for idx_in_other, k in enumerate(other_cols):
            # coefficient for predictor k when target is j
            # In c_all: col 0 = intercept, col idx_in_other+1 = coeff for k
            coeff_sub = c_all[:, idx_in_other + 1]  # (n_sub,)
            coeff_map[(k, j)] = coeff_sub[nearest].astype(np.float32)  # (N,)

    return coeff_map

def build_edge_tensor(df: pd.DataFrame) -> tuple:
    """
    Build edge data tensor and edge types as described in the report.

    For each directed edge k→j (k≠j):
      - Channel 0: observations of variable k, sorted by k
      - Channel 1: observations of variable j, sorted by k
      - Channel 2: multivariate kernel regression coeff c_{i,k→j}, sorted by k

    Returns:
      edge_data:  (p*(p-1), 3, N) float32
      edge_types: (p*(p-1),) int64
    """
    cols = list(df.columns)
    p = len(cols)
    data = df.values.astype(np.float32)  # (N, p)
    N = data.shape[0]

    # Compute ALL multivariate kernel regression coefficients
    coeff_map = compute_multivariate_kernel_coefficients(data)

    edges = []
    edge_types = []

    for i, u_name in enumerate(cols):
        sort_idx = np.argsort(data[:, i])
        u_sorted = data[sort_idx, i]  # (N,)

        for j, v_name in enumerate(cols):
            if i == j:
                continue

            v_sorted_by_u = data[sort_idx, j]               # (N,)
            coeff_sorted = coeff_map[(i, j)][sort_idx]       # (N,)

            edge_tensor = np.stack(
                [u_sorted, v_sorted_by_u, coeff_sorted], axis=0
            )  # (3, N)
            edges.append(edge_tensor)
            edge_types.append(_edge_type(u_name, v_name))

    edge_data = np.stack(edges, axis=0).astype(np.float32)  # (E, 3, N)
    edge_types = np.array(edge_types, dtype=np.int64)        # (E,)
    return edge_data, edge_types

def _build_single(args):
    """Worker function for parallel dataset building."""
    df, y_df = args
    edge_data, edge_types = build_edge_tensor(df)
    result = {
        "edge_data": edge_data,
        "edge_types": edge_types,
        "cols": list(df.columns),
        "p": len(df.columns),
    }
    if y_df is not None:
        result["adj"] = y_df.values.astype(np.float32)
        result["adj_cols"] = list(y_df.columns)
    return result

### Dataset & Collation

In [7]:
class CausalEdgeDataset(Dataset):
    def __init__(self, X_list, y_list=None, n_workers=1):
        args = [
            (X_list[i], y_list[i] if y_list else None)
            for i in range(len(X_list))
        ]
        print(f"Building edge dataset ({len(args)} samples)...")
        # Sequential by default — avoids OOM from process forking
        raw = [_build_single(a) for a in tqdm(args)]

        _, adjacency_label = create_graph_label()
        self.samples = []
        for item in tqdm(raw, desc="Processing samples"):
            sample = {
                "edge_data":  torch.tensor(item["edge_data"]),
                "edge_types": torch.tensor(item["edge_types"]),
                "cols": item["cols"],
                "p": item["p"],
            }
            if "adj" in item:
                adj_np = item["adj"]
                adj_cols = item["adj_cols"]
                sample["adj"] = torch.tensor(adj_np)
                df_adj = pd.DataFrame(adj_np, columns=adj_cols, index=adj_cols)
                node_labels = {}
                for v in adj_cols:
                    if v in ("X", "Y"):
                        continue
                    sub = df_adj.loc[[v, "X", "Y"], [v, "X", "Y"]]
                    key = tuple(sub.values.flatten())
                    label_str = adjacency_label.get(key, "Independent")
                    node_labels[v] = CLASS_NAMES.index(label_str)
                sample["node_labels"] = node_labels
            self.samples.append(sample)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def collate_fn(batch):
    max_E = max(item["edge_data"].shape[0] for item in batch)
    B = len(batch)

    edge_data  = torch.zeros(B, max_E, 3, N_OBS)
    edge_types = torch.zeros(B, max_E, dtype=torch.long)
    edge_mask  = torch.zeros(B, max_E, dtype=torch.bool)
    adj_list, node_labels_list, cols_list = [], [], []

    for b, item in enumerate(batch):
        E = item["edge_data"].shape[0]
        edge_data[b, :E]  = item["edge_data"]
        edge_types[b, :E] = item["edge_types"]
        edge_mask[b, :E]  = True
        cols_list.append(item["cols"])
        if "adj" in item:
            adj_list.append(item["adj"])
            node_labels_list.append(item["node_labels"])

    out = {"edge_data": edge_data, "edge_types": edge_types,
           "edge_mask": edge_mask, "cols": cols_list}
    if adj_list:
        out["adj"] = adj_list
        out["node_labels"] = node_labels_list
    return out

### Model Architecture

Faithful to the report:

1. **StemLayer**: Linear 3 → 64 (channel expansion)
2. **5× ConvBlock**: Conv1d + GroupNorm + GELU + residual
3. **AvgPool** → 64-dim edge embedding
4. **MergeOperator**: concat edge emb + edge type emb → Linear(128→64) + LayerNorm + GELU
5. **2× SelfAttention**: multi-head attention over edges
6. **Edge head**: Linear(64→2) — binary adjacency (training only)
7. **Node head**: Gather 4 edges per node → Merge(4×64→64) → Linear(64→8)

In [8]:
class ConvBlock(nn.Module):
    """Residual Conv1d + GroupNorm + GELU (Fig. 5 in report)."""
    def __init__(self, d, kernel_size=3, n_groups=8):
        super().__init__()
        self.conv = nn.Conv1d(d, d, kernel_size, padding=kernel_size // 2, bias=False)
        self.norm = nn.GroupNorm(n_groups, d)
        self.act = nn.GELU()

    def forward(self, x):
        return x + self.act(self.norm(self.conv(x)))

class MergeOperator(nn.Module):
    """Concat inputs → Linear → LayerNorm → GELU (Fig. 6 in report)."""
    def __init__(self, n_inputs, d):
        super().__init__()
        self.linear = nn.Linear(n_inputs * d, d)
        self.norm = nn.LayerNorm(d)
        self.act = nn.GELU()

    def forward(self, inputs):
        return self.act(self.norm(self.linear(torch.cat(inputs, dim=-1))))

class StemLayer(nn.Module):
    """Linear 3 → d_model applied per-observation."""
    def __init__(self, d):
        super().__init__()
        self.linear = nn.Linear(3, d)

    def forward(self, x):
        # x: (B*E, 3, N) → permute → (B*E, N, 3) → linear → permute back
        return self.linear(x.permute(0, 2, 1)).permute(0, 2, 1)

class EdgeFeatureExtractor(nn.Module):
    """Stem + 5×ConvBlock + AvgPool → 64-dim embedding."""
    def __init__(self, d=64, n_blocks=5):
        super().__init__()
        self.stem = StemLayer(d)
        self.blocks = nn.Sequential(*[ConvBlock(d) for _ in range(n_blocks)])
        self.pool = nn.AdaptiveAvgPool1d(1)

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        return self.pool(x).squeeze(-1)

class SelfAttentionLayer(nn.Module):
    """Standard multi-head self-attention + FFN with pre-norm residuals."""
    def __init__(self, d=64, n_heads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.norm2 = nn.LayerNorm(d)

    def forward(self, x, key_padding_mask=None):
        attn_out, _ = self.attn(x, x, x, key_padding_mask=key_padding_mask)
        x = self.norm1(x + attn_out)
        x = self.norm2(x + self.ff(x))
        return x

class ADIAModel(nn.Module):
    """
    Full model as described in https://thetourney.github.io/adia-report/

    Pipeline (see Fig. 2):
      1. EdgeFeatureExtractor: (B,E,3,N) → (B,E,d)
      2. MergeOperator:        edge emb + type emb → (B,E,d)
      3. 2× SelfAttention:     refine edge embeddings
      4. Edge head:             (B,E,d) → (B,E,2)  [training only]
      5. Node head:             gather 4 edges per node → merge → (B,K,8)
    """
    def __init__(self, d=None, n_edge_types=None):
        super().__init__()
        d = d or D_MODEL
        n_edge_types = n_edge_types or N_EDGE_TYPES

        self.d = d
        self.extractor = EdgeFeatureExtractor(d)
        self.edge_type_emb = nn.Embedding(n_edge_types, d)
        self.edge_merge = MergeOperator(n_inputs=2, d=d)
        self.attn_layers = nn.ModuleList([SelfAttentionLayer(d) for _ in range(2)])
        self.edge_head = nn.Linear(d, 2)
        self.node_merge = MergeOperator(n_inputs=4, d=d)
        self.node_head = nn.Linear(d, N_CLASSES)

    def forward(self, edge_data, edge_types, edge_mask, cols_list):
        B, E, C, N = edge_data.shape

        # --- Feature extraction ---
        edge_emb = self.extractor(edge_data.view(B * E, C, N)).view(B, E, self.d)

        # --- Merge with edge type embeddings ---
        type_emb = self.edge_type_emb(edge_types)
        edge_emb = self.edge_merge([edge_emb, type_emb])

        # --- Self-attention (mask padded edges) ---
        pad_mask = ~edge_mask
        for attn in self.attn_layers:
            edge_emb = attn(edge_emb, key_padding_mask=pad_mask)

        # --- Edge classification head ---
        edge_logits = self.edge_head(edge_emb)  # (B, E, 2)

        # --- Node classification head ---
        node_logits_list = []
        for b in range(B):
            cols = cols_list[b]
            p = len(cols)
            col_idx = {name: i for i, name in enumerate(cols)}

            # Build edge index map: (u_idx, v_idx) → position in edge list
            edge_order = {}
            count = 0
            for ui in range(p):
                for vi in range(p):
                    if ui != vi:
                        edge_order[(ui, vi)] = count
                        count += 1

            x_idx = col_idx.get("X")
            y_idx = col_idx.get("Y")
            embs = edge_emb[b]
            other_nodes = [n for n in cols if n not in ("X", "Y")]

            if not other_nodes or x_idx is None or y_idx is None:
                node_logits_list.append(None)
                continue

            node_logits = []
            for node_name in other_nodes:
                u = col_idx[node_name]
                # Gather 4 edges: u→X, u→Y, X→u, Y→u (Fig. 7)
                gathered = [
                    embs[edge_order[(u, x_idx)]],
                    embs[edge_order[(u, y_idx)]],
                    embs[edge_order[(x_idx, u)]],
                    embs[edge_order[(y_idx, u)]],
                ]
                node_emb = self.node_merge(gathered)
                node_logits.append(self.node_head(node_emb))

            node_logits_list.append(
                torch.stack(node_logits) if node_logits else None
            )

        return edge_logits, node_logits_list

### Training Wrapper

Loss = CE(edge classification) + CE(node classification),
both weighted by inverse class frequency as described in the report.

In [9]:
def compute_class_weights(y_list):
    """Inverse-frequency weights for the 8 node classes."""
    adjacency_label = get_adjacency_label()
    counts = torch.zeros(N_CLASSES)
    for y_df in y_list:
        cols = list(y_df.columns)
        arr = y_df.values
        col_idx = {c: i for i, c in enumerate(cols)}
        for v in cols:
            if v in ("X", "Y"):
                continue
            idx = [col_idx[v], col_idx["X"], col_idx["Y"]]
            sub = arr[np.ix_(idx, idx)]
            key = tuple(sub.flatten())
            label_str = adjacency_label.get(key, "Independent")
            counts[CLASS_NAMES.index(label_str)] += 1
    w = 1.0 / (counts + 1e-6)
    return w / w.sum() * N_CLASSES

def compute_edge_class_weights(y_list):
    """Inverse-frequency weights for binary edge classes."""
    counts = torch.zeros(2)
    for y_df in y_list:
        arr = y_df.values
        p = arr.shape[0]
        for i in range(p):
            for j in range(p):
                if i != j:
                    counts[int(arr[i, j])] += 1
    w = 1.0 / (counts + 1e-6)
    return w / w.sum() * 2

class ADIAModelWrapper(pl.LightningModule):
    def __init__(self, d=None, node_class_weights=None,
                 edge_class_weights=None, lr=1e-3, max_epochs=20):
        super().__init__()
        d = d or D_MODEL

        self.model = ADIAModel(d)
        self.lr = lr
        self.max_epochs = max_epochs
        self.node_criterion = nn.CrossEntropyLoss(
            weight=node_class_weights if node_class_weights is not None
            else torch.ones(N_CLASSES)
        )
        self.edge_criterion = nn.CrossEntropyLoss(
            weight=edge_class_weights if edge_class_weights is not None
            else torch.ones(2)
        )

    def forward(self, batch):
        return self.model(
            batch["edge_data"].to(self.device),
            batch["edge_types"].to(self.device),
            batch["edge_mask"].to(self.device),
            batch["cols"],
        )

    def _compute_loss(self, batch, split):
        edge_logits, node_logits_list = self(batch)
        B = edge_logits.shape[0]
        cols_list = batch["cols"]
        adj_list = batch.get("adj")
        node_labels_list = batch.get("node_labels")

        total_loss = torch.tensor(0.0, device=self.device)
        n_terms = 0

        for b in range(B):
            cols = cols_list[b]
            p = len(cols)
            E_b = p * (p - 1)

            # Edge loss
            if adj_list is not None:
                adj = adj_list[b].to(self.device)
                edge_labels = []
                for ui in range(p):
                    for vi in range(p):
                        if ui != vi:
                            edge_labels.append(int(adj[ui, vi].item()))
                edge_labels = torch.tensor(edge_labels, dtype=torch.long, device=self.device)
                total_loss = total_loss + self.edge_criterion(
                    edge_logits[b, :E_b], edge_labels
                )
                n_terms += 1

            # Node loss
            if node_labels_list is not None and node_logits_list[b] is not None:
                labels_dict = node_labels_list[b]
                other = [n for n in cols if n not in ("X", "Y")]
                labels_t = torch.tensor(
                    [labels_dict[n] for n in other],
                    dtype=torch.long, device=self.device
                )
                total_loss = total_loss + self.node_criterion(
                    node_logits_list[b], labels_t
                )
                n_terms += 1

        if n_terms > 0:
            total_loss = total_loss / n_terms
        self.log(f"{split}_loss", total_loss, on_step=(split == "train"),
                 on_epoch=True, prog_bar=True, batch_size=B)
        return total_loss

    def training_step(self, batch, batch_idx):
        return self._compute_loss(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self._compute_loss(batch, "val")

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.max_epochs)
        return [opt], [{"scheduler": sched, "interval": "epoch"}]

### Inference

In [10]:
@torch.no_grad()
def infer_single(df, model, device="cpu"):
    """Run inference on a single graph, return adjacency DataFrame."""
    model = model.eval()
    cols = list(df.columns)
    p = len(cols)

    edge_data, edge_types = build_edge_tensor(df)
    edge_data_t  = torch.tensor(edge_data).unsqueeze(0).to(device)
    edge_types_t = torch.tensor(edge_types).unsqueeze(0).to(device)
    edge_mask_t  = torch.ones(1, len(edge_types), dtype=torch.bool, device=device)

    edge_logits, node_logits_list = model(
        edge_data_t, edge_types_t, edge_mask_t, [cols]
    )

    # Edge probs → DAG
    edge_probs = torch.softmax(edge_logits[0], dim=-1)[:, 1]
    E_mat = np.zeros((p, p))
    count = 0
    for i in range(p):
        for j in range(p):
            if i != j:
                E_mat[i, j] = edge_probs[count].item()
                count += 1

    adj = transform_proba_to_DAG(cols, E_mat).astype(int)
    A = pd.DataFrame(adj, columns=cols, index=cols)

    # Override with node classification head
    if node_logits_list[0] is not None:
        other_nodes = [n for n in cols if n not in ("X", "Y")]
        node_preds = torch.argmax(node_logits_list[0], dim=-1)
        patterns = {
            "Confounder":        lambda n: [(n, "X"), (n, "Y")],
            "Collider":          lambda n: [("X", n), ("Y", n)],
            "Mediator":          lambda n: [("X", n), (n, "Y")],
            "Cause of X":        lambda n: [(n, "X")],
            "Cause of Y":        lambda n: [(n, "Y")],
            "Consequence of X":  lambda n: [("X", n)],
            "Consequence of Y":  lambda n: [("Y", n)],
            "Independent":       lambda n: [],
        }
        for k, node_name in enumerate(other_nodes):
            pred_class = CLASS_NAMES[node_preds[k].item()]
            A.loc[node_name, :] = 0
            A.loc[:, node_name] = 0
            for (src, dst) in patterns[pred_class](node_name):
                A.loc[src, dst] = 1

    return A

### CrunchDAO Interface — `train` & `infer`

In [11]:
# @crunch/keep:on
MAX_EPOCHS = 30

def train(
    X_train: typing.Dict[str, pd.DataFrame],
    y_train: typing.Dict[str, pd.DataFrame],
    model_directory_path: str,
) -> None:
    keys = list(X_train.keys())
    X_list = [X_train[k] for k in keys]
    y_list = [y_train[k] for k in keys]

    node_w = compute_class_weights(y_list)
    edge_w = compute_edge_class_weights(y_list)
    dataset = CausalEdgeDataset(X_list, y_list)

    wrapper = ADIAModelWrapper(
        d=D_MODEL, node_class_weights=node_w,
        edge_class_weights=edge_w, lr=1e-3, max_epochs=MAX_EPOCHS,
    )
    trainer = pl.Trainer(
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        devices=1, max_epochs=MAX_EPOCHS, precision="16-mixed",
        logger=False, enable_checkpointing=False, enable_progress_bar=True,
    )
    loader = DataLoader(
        dataset, batch_size=16, shuffle=True, drop_last=True,
        num_workers=0, collate_fn=collate_fn,
    )
    trainer.fit(wrapper, loader)

    path = os.path.join(model_directory_path, "model.pt")
    torch.save(wrapper.model.state_dict(), path)
    print(f"Model saved to {path}")

def infer(
    X_test: typing.Dict[str, pd.DataFrame],
    model_directory_path: str,
    id_column_name: str,
    prediction_column_name: str,
) -> pd.DataFrame:
    path = os.path.join(model_directory_path, "model.pt")
    model = ADIAModel(d=D_MODEL)
    model.load_state_dict(torch.load(path, map_location="cpu", weights_only=True))
    model.to("cpu")
    model.eval()

    submission = {}
    for name in tqdm(X_test, desc="Inferring"):
        df = X_test[name]
        nodes = list(df.columns)
        A = infer_single(df, model, "cpu")
        for i in nodes:
            for j in nodes:
                submission[f"{name}_{i}_{j}"] = int(A.loc[i, j])

    s = pd.Series(submission).reset_index()
    s.columns = [id_column_name, prediction_column_name]
    return s

### Local Training & Evaluation

In [12]:
import crunch

crunch = crunch.load_notebook()

loaded inline runner with module: <module '__main__'>

cli version: 11.0.0
available ram: 31.00 gb
available cpu: 8 core
----


In [13]:
# X_train, y_train, X_test = crunch.load_data()

In [14]:
# from sklearn.metrics import balanced_accuracy_score

# keys = list(X_train.keys())
# train_keys, test_keys = train_test_split(keys, test_size=0.2, random_state=42)

# def evaluate_local(X_train, y_train, max_samples=200, max_epochs=20):
#     eval_train_keys = train_keys[:max_samples * 10]
#     eval_test_keys = test_keys[:max_samples]

#     X_tr = [X_train[k] for k in eval_train_keys]
#     y_tr = [y_train[k] for k in eval_train_keys]

#     node_w = compute_class_weights(y_tr)
#     edge_w = compute_edge_class_weights(y_tr)
#     dataset = CausalEdgeDataset(X_tr, y_tr)

#     wrapper = ADIAModelWrapper(
#         d=D_MODEL, node_class_weights=node_w,
#         edge_class_weights=edge_w, lr=1e-3, max_epochs=max_epochs,
#     )
#     trainer = pl.Trainer(
#         accelerator="gpu" if torch.cuda.is_available() else "cpu",
#         devices=1, max_epochs=max_epochs, precision="16-mixed",
#         logger=False, enable_checkpointing=False, enable_progress_bar=True,
#     )
#     loader = DataLoader(
#         dataset, batch_size=16, shuffle=True, drop_last=True,
#         num_workers=0, collate_fn=collate_fn,
#     )
#     trainer.fit(wrapper, loader)

#     # Inference
#     device = "cuda" if torch.cuda.is_available() else "cpu"
#     model = wrapper.model.to(device).eval()
#     adjacency_label = get_adjacency_label()
#     y_pred, y_true = [], []
#     for name in tqdm(eval_test_keys, desc="Evaluating"):
#         A = infer_single(X_train[name], model, device)
#         predicted = get_labels(A, adjacency_label)
#         ground_truth = get_labels(y_train[name], adjacency_label)
#         for k in predicted:
#             y_pred.append(predicted[k])
#             y_true.append(ground_truth[k])

#     y_pred, y_true = pd.Series(y_pred), pd.Series(y_true)
#     scores = {}
#     for label in y_true.unique():
#         scores[label] = np.mean(y_pred[y_true == label] == label)
#     scores = pd.Series(scores)
#     scores["Balanced Accuracy"] = balanced_accuracy_score(y_true, y_pred)
#     return scores

In [15]:
# scores = evaluate_local(X_train, y_train, max_samples=500, max_epochs=5)
# print(scores)

### CrunchDAO Test & Submit

In [16]:
crunch.test(no_determinism_check=True)

print("Submit at: https://hub.crunchdao.com/competitions/causality-discovery/submit/via/notebook")

11:45:13 


11:45:14 started


11:45:14 running local test


11:45:14 internet access isn't restricted, no check will be done


11:45:14 


...

17:02:03 save prediction - path=prediction


17:02:03 ended


17:02:03 duration - time=05:16:48


17:02:03 memory - before="887.77 MB" after="19.47 GB" consumed="18.59 GB"


Submit at: https://hub.crunchdao.com/competitions/causality-discovery/submit/via/notebook


# Calculate accuracy

In [ ]:
# import pandas as pd
# from sklearn.metrics import balanced_accuracy_score

In [ ]:
# def calculate_submission(y_true: pd.Series, y_pred: pd.Series) -> float:
#     return balanced_accuracy_score(y_true, y_pred)

# y_true = infer(X_test, model_directory_path="./resources", id_column_name="id", prediction_column_name="prediction")

In [19]:
# y_true

In [20]:
# y_pred = pd.read_parquet("data/example_prediction_reduced.parquet")
# y_pred

In [21]:
# calculate_submission(y_true["prediction"], y_pred["prediction"])